# Chapter 28: Sequence Models and Transformers

## Learning Objectives\n- Understand scaled dot-product attention\n- Compare RNN/LSTM vs transformer assumptions\n- Build single-head attention in pure Python\n- Plan practical fine-tuning workflow

## Prerequisites\n- Python basics and functions\n- Numpy basics for deep learning chapters\n- Understanding of earlier chapters (0-19)\n

# Chapter 28: Sequence Models and Transformers

## Why this chapter matters
Most modern NLP and many recommendation/time-series systems rely on transformer-style attention.

## Learning goals
1. Understand self-attention math.
2. Compare RNN/LSTM vs transformer tradeoffs.
3. Implement scaled dot-product attention.
4. Learn practical fine-tuning patterns.

## Step 1: Attention equation
\[
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

## Step 2: Transformer block components
1. multi-head self-attention
2. residual + layer norm
3. feed-forward network
4. residual + layer norm

## Step 3: Training considerations
- tokenization quality is critical
- warmup + decay LR schedules
- gradient clipping for stability
- mixed precision for throughput

## Step 4: Fine-tuning strategy
1. start with pretrained checkpoint
2. tune only head first
3. gradual unfreeze if needed
4. evaluate by business slices, not only global accuracy

## Assignment
Implement single-head attention from scratch and verify outputs on toy tokens.
Then build next-token toy predictor using your own tokenized corpus.



In [ ]:
from __future__ import annotations

import math
from typing import List


Vector = List[float]
Matrix = List[Vector]


def transpose(m: Matrix) -> Matrix:
    return [list(col) for col in zip(*m)]


def matmul(a: Matrix, b: Matrix) -> Matrix:
    bt = transpose(b)
    out = []
    for row in a:
        out_row = []
        for col in bt:
            out_row.append(sum(x * y for x, y in zip(row, col)))
        out.append(out_row)
    return out


def softmax_row(row: Vector) -> Vector:
    m = max(row)
    ex = [math.exp(x - m) for x in row]
    z = sum(ex)
    return [v / z for v in ex]


def attention(Q: Matrix, K: Matrix, V: Matrix) -> Matrix:
    dk = len(K[0])
    scores = matmul(Q, transpose(K))
    scores = [[x / math.sqrt(dk) for x in row] for row in scores]
    weights = [softmax_row(row) for row in scores]
    return matmul(weights, V)


if __name__ == "__main__":
    Q = [[1.0, 0.0], [0.0, 1.0]]
    K = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
    V = [[1.0, 2.0], [0.0, 1.0], [3.0, 1.0]]

    out = attention(Q, K, V)
    print("Attention output:")
    for row in out:
        print([round(v, 4) for v in row])


## Checkpoint\n\n1. You can explain Q, K, V roles.\n2. You can compute attention weights and output.\n3. You can describe why scaling by sqrt(dk) is used.

## Guided Exercise\nAdd a simple causal mask so token `t` cannot attend to future tokens.

In [ ]:
# TODO (Student Exercise)
# Add masking support in attention() by setting masked scores to a large negative value before softmax.
pass

## Quick Quiz\n\n**Q1.** What does self-attention optimize in sequence modeling?\n\n**Answer:** Context-aware token representation using pairwise relevance.\n\n**Q2.** Why layer normalization and residuals?\n\n**Answer:** They stabilize and speed deep optimization.\n\n**Q3.** When might RNN still be chosen?\n\n**Answer:** For tiny models or strict streaming constraints.\n

## Homework\nImplement multi-head attention by splitting embedding dimension into multiple heads.